# 13주차 · 네트워크 분석

**⏱ 60분** · 15세기 피렌체 15개 가문의 혼인 관계

## 🎯 이번 주의 질문

1427년 피렌체에서 재산(146 대 103, 천 리라)과 시의회 의석(74 대 53)은 Strozzi 가문이 Medici 가문보다 많았다. 그런데 1434년 이후 피렌체를 이끈 가문은 Medici였다. Padgett·Ansell(1993)은 가문들 사이의 혼인 관계를 네트워크로 놓고 Medici가 그 안에서 어떤 위치에 있었는지로 이것을 설명했다.

> **혼인 네트워크에서 Medici는 Strozzi와 어떻게 다른 위치에 있었나?**

**데이터:** Padgett·Ansell(1993)의 피렌체 가문 혼인 관계. 가문 15개, 혼인 관계 20개.

| 셀 표시 | 할 일 |
|---|---|
| `# ✍️ 직접 입력` | 위에 보이는 코드를 직접 쳐서 실행한다 |
| `# ▶ 그대로 실행` | 준비·그림 코드이다. 실행만 한다 |

> **실습본이다.** 먼저 메뉴 **파일 → Drive에 사본 저장**을 누른다. 사본을 저장하지 않으면 입력한 코드가 사라진다.
> 맨 위 준비 셀부터 순서대로 `Shift + Enter`로 실행한다. ▶ 셀은 그대로 실행하고, ✍️ 셀은 위에 보여 준 코드를
> 빈 셀에 손으로 타이핑해 실행한다.

In [ ]:
# ▶ 준비 셀 — 그대로 실행
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
print("준비 완료!", nx.__version__)

---
## 1. link 목록: 네트워크 데이터의 기본 모양

네트워크 데이터는 한 행이 관계 하나인 표로 적는다. 이 표를 link 목록(edge list)이라 한다. 혼인은 두 가문이 함께 맺는 관계라 방향이 없고, 혼인이 있었는지만 기록하므로 가중치도 없다. 20개 혼인 관계를 두 열짜리 표로 적는다.

In [ ]:
# ▶ 그대로 실행 — 20개 혼인 관계의 link 목록
df = pd.DataFrame([
    ("Acciaiuoli", "Medici"), ("Albizzi", "Ginori"), ("Albizzi", "Guadagni"), ("Albizzi", "Medici"),
    ("Barbadori", "Castellani"), ("Barbadori", "Medici"), ("Bischeri", "Guadagni"), ("Bischeri", "Peruzzi"),
    ("Bischeri", "Strozzi"), ("Castellani", "Peruzzi"), ("Castellani", "Strozzi"), ("Guadagni", "Lamberteschi"),
    ("Guadagni", "Tornabuoni"), ("Medici", "Ridolfi"), ("Medici", "Salviati"), ("Medici", "Tornabuoni"),
    ("Pazzi", "Salviati"), ("Peruzzi", "Strozzi"), ("Ridolfi", "Strozzi"), ("Ridolfi", "Tornabuoni"),
], columns=["family_a", "family_b"])
df.head()

link 목록을 네트워크로 바꾼다. 두 열의 이름을 알려 주면 행마다 link 하나를 만든다.

```python
G = nx.from_pandas_edgelist(df, "family_a", "family_b")
G.number_of_nodes(), G.number_of_edges()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 **결과 해설** — node 15개, link 20개다. ✅ `(15, 20)`

---
## 2. 그림과 인접 행렬

같은 네트워크를 그림으로 그린다.

In [ ]:
# ▶ 그대로 실행 — 혼인 네트워크 그림
pos = nx.kamada_kawai_layout(G)
plt.figure(figsize=(8, 6))
nx.draw_networkx(G, pos, node_color="lightsteelblue", node_size=900, font_size=9)
plt.axis("off")
plt.show()

인접 행렬(adjacency matrix)은 행과 열이 모두 node이고, 두 node 사이에 link가 있으면 1, 없으면 0인 표다.

```python
A = nx.to_pandas_adjacency(G, dtype=int)
A.loc[["Medici", "Strozzi", "Salviati"]]
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 **결과 해설** — Medici 행에는 1이 6개, Strozzi 행에는 4개 있다. 행의 합이 그 가문에 이어진 혼인의 수다.

---
## 3. centrality

### degree centrality

degree centrality는 node에 직접 이어진 link의 수다.

```python
degree = pd.Series(dict(G.degree())).sort_values(ascending=False)
degree.head()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 **결과 해설** — Medici 6, Strozzi와 Guadagni 4다. ✅ Medici 6

### betweenness centrality

betweenness centrality는 다른 두 node 사이의 최단 경로가 그 node를 얼마나 지나는지 나타낸다. 이론과 같은 값으로 보기 위해 `normalized=False`로 나누지 않은 값을 구한다.

```python
betweenness = pd.Series(nx.betweenness_centrality(G, normalized=False)).sort_values(ascending=False)
betweenness.round(2).head(7)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 **결과 해설** — Medici 47.50으로 2위 Guadagni(23.17)의 두 배가 넘는다. Strozzi는 9.33이다. 이론에서 손으로 센 Salviati는 13.00이다. ✅ Medici 47.50, Salviati 13.00

---
## 4. robustness: 가문 하나를 빼면

$r_i$는 node $i$를 뺀 뒤 가장 큰 connected component에 속하지 못한 node의 수다. 가문 하나씩 빼 보며 계산한다.

```python
n = G.number_of_nodes()
robustness = {}
for family in G.nodes():
    H = G.copy()
    H.remove_node(family)
    largest = max(nx.connected_components(H), key=len)
    robustness[family] = (n - 1) - len(largest)
robustness = pd.Series(robustness).sort_values(ascending=False)
robustness.head()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 **결과 해설** — Medici를 빼면 3개 가문(Acciaiuoli, Salviati, Pazzi)이 떨어져 나간다. betweenness 2위 Guadagni를 빼면 Lamberteschi 하나만 떨어진다. ✅ Medici 3

---
## 5. community: 가문들을 그룹으로 나누기

greedy modularity algorithm은 modularity $Q$가 가장 크게 늘어나는 두 그룹을 차례로 합친다.

```python
groups = nx.community.greedy_modularity_communities(G)
for i, group in enumerate(groups, 1):
    print("그룹", i, sorted(group))
round(nx.community.modularity(G, groups), 5)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


In [ ]:
# ▶ 그대로 실행 — 그룹별 색, 점의 크기는 betweenness centrality
color = {f: i for i, group in enumerate(groups) for f in group}
plt.figure(figsize=(8, 6))
nx.draw_networkx(G, pos, node_color=[color[f] for f in G.nodes()], cmap="Set2",
                 node_size=[300 + 40 * betweenness[f] for f in G.nodes()], font_size=9)
plt.axis("off")
plt.show()

💬 **결과 해설** — 그룹 3개, $Q$ = 0.39875다. Medici는 Acciaiuoli·Pazzi·Ridolfi·Salviati·Tornabuoni와 한 그룹이고, Strozzi는 Barbadori·Bischeri·Castellani·Peruzzi와 다른 그룹이다. ✅ 그룹 3개, $Q$ = 0.39875

---
## 6. Medici와 Strozzi

지금까지 구한 값을 두 가문만 모아 본다.

```python
pd.DataFrame({"degree": degree, "betweenness": betweenness.round(2), "robustness": robustness}).loc[["Medici", "Strozzi"]]
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 **결과 해설** — Medici는 degree 6, betweenness 47.50이고, 빼면 3개 가문이 떨어져 나간다. Strozzi는 degree 4, betweenness 9.33이고, 빼도 떨어져 나가는 가문이 없다. 재산과 의석은 Strozzi가 많았지만, 서로 이어지지 않은 가문들 사이를 잇는 자리에는 Medici가 있었다.

## ✅ 정리

| 단계 | 코드 | 이 자료의 값 |
|---|---|---|
| link 목록 → 네트워크 | `nx.from_pandas_edgelist(df, "family_a", "family_b")` | node 15, link 20 |
| 인접 행렬 | `nx.to_pandas_adjacency(G, dtype=int)` | Medici 행의 합 6 |
| degree centrality | `dict(G.degree())` | Medici 6 |
| betweenness centrality | `nx.betweenness_centrality(G, normalized=False)` | Medici 47.50 |
| robustness | node를 빼고 가장 큰 connected component의 크기 | Medici 3 |
| community | `nx.community.greedy_modularity_communities(G)` | 그룹 3개, $Q$ = 0.39875 |

랩에서는 같은 순서를 2022 카타르 월드컵 한 팀의 패스 기록에 적용한다.